In [1]:
import pandas as pd
import re
import os

# ============================================================
# 1. 设置数据文件夹路径
# ============================================================

folder_path = r'C:\Users\choos\Desktop\5508-剧本数据清洗\标注\标注版'


# ============================================================
# 2. 自动扫描四个阶段的 CSV 文件
# ============================================================

file_paths = {}

try:
    for filename in os.listdir(folder_path):
        if '01_' in filename and filename.endswith('.csv'):
            file_paths['01_引流'] = os.path.join(folder_path, filename)
        elif '02_' in filename and filename.endswith('.csv'):
            file_paths['02_建立信任'] = os.path.join(folder_path, filename)
        elif '03_' in filename and filename.endswith('.csv'):
            file_paths['03_收割'] = os.path.join(folder_path, filename)
        elif '04_' in filename and filename.endswith('.csv'):
            file_paths['04_工具'] = os.path.join(folder_path, filename)

except FileNotFoundError:
    print(f"未找到文件夹，请检查路径：{folder_path}")

print("正在扫描 CSV 文件...")
for stage, path in file_paths.items():
    print(f"已找到文件：[{stage}] -> {path}")
print("=" * 50)


# ============================================================
# 3. 读取并合并四个阶段的数据
# ============================================================

dfs = []

for stage, path in file_paths.items():
    try:
        # 优先使用 UTF-8 编码读取文件
        df = pd.read_csv(path, encoding='utf-8')
        df['诈骗阶段'] = stage
        dfs.append(df)
        print(f"成功读取 {stage} 文件，编码：UTF-8，记录数：{len(df)}")

    except UnicodeDecodeError:
        # 如果 UTF-8 读取失败，则使用 GB18030 编码读取中文文件
        try:
            df = pd.read_csv(path, encoding='gb18030')
            df['诈骗阶段'] = stage
            dfs.append(df)
            print(f"成功读取 {stage} 文件，编码：GB18030，记录数：{len(df)}")

        except Exception as e2:
            print(f"读取 {stage} 文件失败，错误信息：{e2}")

    except Exception as e:
        print(f"读取 {stage} 文件失败，错误信息：{e}")


# ============================================================
# 4. 描述性统计分析
# ============================================================

if len(dfs) == 4:
    all_data = pd.concat(dfs, ignore_index=True)
    print(f"\n数据合并完成，总话术记录数：{len(all_data)}\n")

    # ------------------------------------------------------------
    # 4.1 从“提及金额”字段中提取最大金额
    # ------------------------------------------------------------

    def extract_max_amount(val):
        if pd.isna(val) or str(val).strip() == '':
            return 0

        nums = re.findall(r'\d+\.?\d*', str(val).replace(',', ''))
        return max([float(n) for n in nums]) if nums else 0

    if '提及金额' in all_data.columns:
        all_data['最大提取金额'] = all_data['提及金额'].apply(extract_max_amount)
    else:
        all_data['最大提取金额'] = 0


    # ------------------------------------------------------------
    # 4.2 分析不同诈骗阶段中的诈骗人设分布
    # ------------------------------------------------------------

    if '诈骗人设_API' in all_data.columns:
        print("分析一：不同诈骗阶段的诈骗人设分布占比（%）")

        persona_pivot = pd.crosstab(
            all_data['诈骗阶段'],
            all_data['诈骗人设_API'],
            normalize='index'
        ) * 100

        persona_pivot = persona_pivot.round(1)
        print(persona_pivot)
        print("-" * 50)

        persona_pivot.to_csv(
            os.path.join(folder_path, '板块1_诈骗人设演变.csv'),
            encoding='utf-8-sig'
        )


    # ------------------------------------------------------------
    # 4.3 分析不同诈骗阶段中的金额与心理策略变化
    # ------------------------------------------------------------

    psy_cols = [col for col in all_data.columns if '心理策略' in col]

    if psy_cols:
        agg_dict = {'最大提取金额': 'mean'}

        for col in psy_cols:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            agg_dict[col] = 'mean'

        stage_analysis = all_data.groupby('诈骗阶段').agg(agg_dict).reset_index()

        # 将平均金额与心理策略比例格式化，方便查看和导出
        stage_analysis['最大提取金额'] = (
            stage_analysis['最大提取金额'].round(2).astype(str) + " 元"
        )

        for col in psy_cols:
            stage_analysis[col] = (
                (stage_analysis[col] * 100).round(1).astype(str) + "%"
            )

        print("\n分析二：不同诈骗阶段的涉案金额与核心心理策略变化")
        print(stage_analysis.to_string())

        stage_analysis.to_csv(
            os.path.join(folder_path, '板块2_金额与心理策略演变.csv'),
            encoding='utf-8-sig',
            index=False
        )

    print("\n分析完成，结果文件已保存至原数据文件夹。")

else:
    print(f"\n预期读取 4 个阶段文件，实际成功读取 {len(dfs)} 个文件。")

正在扫描 CSV 文件...
已找到文件：[01_引流] -> C:\Users\choos\Desktop\5508-剧本数据清洗\标注\标注版\_合并_01_准备与引流(1)_仅追加标注145.csv
已找到文件：[02_建立信任] -> C:\Users\choos\Desktop\5508-剧本数据清洗\标注\标注版\_合并_02_建立信任与诱导(1)_仅追加标注145.csv
已找到文件：[03_收割] -> C:\Users\choos\Desktop\5508-剧本数据清洗\标注\标注版\_合并_03_交易与收割(1)_仅追加标注145.csv
已找到文件：[04_工具] -> C:\Users\choos\Desktop\5508-剧本数据清洗\标注\标注版\_合并_04_辅助知识与工具(1)_仅追加标注145.csv
成功读取 01_引流 文件，编码：UTF-8，记录数：216
成功读取 02_建立信任 文件，编码：UTF-8，记录数：833
成功读取 03_收割 文件，编码：UTF-8，记录数：180
成功读取 04_工具 文件，编码：UTF-8，记录数：429

数据合并完成，总话术记录数：1658

分析一：不同诈骗阶段的诈骗人设分布占比（%）
诈骗人设_API  内部人   同伴  官方客服    恋人  成功人设  投资导师
诈骗阶段                                      
01_引流     5.1  0.9  34.4  12.6  27.0  20.0
02_建立信任   4.2  1.3   9.6  29.9  27.9  27.1
03_收割     2.2  0.0  14.4  14.4  23.3  45.6
04_工具     8.2  0.9  14.7  14.7  34.3  27.3
--------------------------------------------------

分析二：不同诈骗阶段的涉案金额与核心心理策略变化
      诈骗阶段        最大提取金额 心理策略_贪婪 心理策略_恐惧 心理策略_信任权威 心理策略_情感依赖 心理策略_从众 心理策略_紧迫感
0    01_引流    71245.69 元   58.8%   27.8%    